# Solar filament instance segmentation - reproducible pipeline

Filament Segmentation Challenge 2026 - Doğukan Turan. Companion of the 4-page report; every module is documented in `README.md`.

```
H-alpha photograph -> 3-channel input -> 10 U-Nets x 8 flips -> probability maps (foreground, border)
                   -> candidate instances (threshold 0.5, link gaps < 14 px) -> learned score g_hat per candidate
                   -> keep g_hat >= 0.20 -> disjoint instance masks -> submission.csv (COCO RLE, 2048 x 2048)
```

**Modes** (`RUN_MODE`):

* `REPRODUCE_SUBMISSION` (default) - uses the 10 trained networks stored in `weights/` (SHA-256 checked; a missing file is downloaded from the URL in `configs/weights.json`), predicts the 180 test photographs and writes `submission.csv`. No training. About one GPU hour (T4 / P100); `QUICK = True` handles six photographs in a few minutes.
* `REFIT_FILTER` - regenerates the out-of-fold probability maps from the stored networks (each predicts its held-out fold; about one GPU hour), refits the learned instance filter on them, recomputes the ablation table of the report (`ablate`, with day-level bootstrap intervals) and predicts the test set with the refitted filter.
* `TRAIN_FULL` - retrains the 10 networks (5 folds x {1536, 2048}; 20+ GPU hours), fits the learned instance filter on their out-of-fold predictions, evaluates it, then predicts the test set.

At test time the only input is the test photograph itself: no metadata, neighbouring observations, ground truth or external labels.

In [ ]:
RUN_MODE = "REPRODUCE_SUBMISSION"   # or "REFIT_FILTER" or "TRAIN_FULL"
DATA_ROOT = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"   # folder that contains train/ and test/
WORK = "work"        # caches, probability maps, intermediate files
TTA = 8              # dihedral test-time augmentations (1, 2, 4 or 8)
QUICK = False        # True: only the first 6 test photographs (environment check, a few minutes)
INSTALL = "auto"     # "auto": pip install -r requirements.txt only if a required package cannot be imported; True: always; False: never

In [ ]:
import glob, importlib.util, os, subprocess, sys

REQUIRED = ["torch", "torchvision", "segmentation_models_pytorch", "timm", "cv2", "pycocotools", "pandas", "scipy", "skimage", "sklearn", "matplotlib", "PIL"]
absent = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if INSTALL is True or (INSTALL == "auto" and absent):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("pip exit code", r.returncode, "(non-zero: the packages that are already installed are used)")
else:
    print("environment:", "all required packages found" if not absent else f"missing {absent}: set INSTALL = 'auto' or True")

import segmentation_models_pytorch as smp
print("segmentation-models-pytorch", smp.__version__, "(the stored checkpoints were trained with 0.5.0)")

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only (inference will be slow)")


def run(*args):
    """Run `python -m <module> ...` of this repository; the same commands are listed in the README."""
    print("$ python -m", *map(str, args), flush=True)
    subprocess.run([sys.executable, "-m", *map(str, args)], check=True)


LIMIT = ["--limit", 6] if QUICK else []

## 1. Models and learned instance filter

`REPRODUCE_SUBMISSION`: the ten checkpoints (EfficientNet-B4 U-Net, EMA weights, 5 folds x 2 resolutions) and the training table of the learned filter (`configs/rescorer_all.npz`, re-fitted deterministically in seconds).

`REFIT_FILTER`: the out-of-fold maps are regenerated from the stored checkpoints (`oof`) and the filter is fitted on them (`rescore`).

`TRAIN_FULL`: five folds grouped by observation day at both resolutions; every fold also writes the out-of-fold probability maps of its held-out photographs. The filter is then fitted on the out-of-fold candidates only (`rescore`).

In [ ]:
if RUN_MODE == "REPRODUCE_SUBMISSION":
    run("solarfil.weights", "--manifest", "configs/weights.json", "--dest", "weights")   # no download when weights/ is intact
    CKPTS, RESCORER = sorted(glob.glob("weights/Unet_*.pth")), "configs/rescorer_all.npz"
elif RUN_MODE == "REFIT_FILTER":
    run("solarfil.weights", "--manifest", "configs/weights.json", "--dest", "weights")
    CKPTS = sorted(glob.glob("weights/Unet_*.pth"))
    run("solarfil.oof", "--data_root", DATA_ROOT, "--ckpts", *CKPTS, "--out", WORK)
    oof = [f"{WORK}/oof_Unet_efficientnet-b4_s{s}" for s in (1536, 2048)]
    run("solarfil.rescore", "--data_root", DATA_ROOT, "--postcfg", "configs/postcfg.json", "--gt", "all=1,2,3,4", "--thr", 0.20,
        "--view", oof[0], "--view", ",".join(oof), "--out", f"{WORK}/rescorer_{{name}}.npz")
    RESCORER = f"{WORK}/rescorer_all.npz"
elif RUN_MODE == "TRAIN_FULL":
    for size, epochs in ((1536, 36), (2048, 32)):
        for fold in range(5):
            run("solarfil.train", "--data_root", DATA_ROOT, "--work", WORK, "--fold", fold, "--size", size, "--crop", 768, "--epochs", epochs)
    oof = [f"{WORK}/oof_Unet_efficientnet-b4_s{s}" for s in (1536, 2048)]
    run("solarfil.rescore", "--data_root", DATA_ROOT, "--postcfg", "configs/postcfg.json", "--gt", "all=1,2,3,4", "--thr", 0.20,
        "--view", oof[0], "--view", ",".join(oof), "--out", f"{WORK}/rescorer_{{name}}.npz")
    CKPTS, RESCORER = sorted(glob.glob(f"{WORK}/Unet_*.pth")), f"{WORK}/rescorer_all.npz"
else:
    raise ValueError(RUN_MODE)
print(len(CKPTS), "checkpoints | learned filter:", RESCORER)

### Out-of-fold evaluation (`REFIT_FILTER` and `TRAIN_FULL`)

Cross-fitted candidate scores of the out-of-fold photographs, scored with the re-implementation of the official metric (PQ pooled over annotator copies, IoU > 0.5) and the rubric measures; intervals resample observation days.

In [ ]:
if RUN_MODE in ("REFIT_FILTER", "TRAIN_FULL"):
    run("solarfil.cands", "--data_root", DATA_ROOT, "--rescorer", RESCORER, "--view", oof[0], "--view", ",".join(oof), "--out", f"{WORK}/cands_oof.pkl")
    run("solarfil.cands", "--emit", f"{WORK}/cands_oof.pkl", "--thr", 0.20, "--out", f"{WORK}/oof_predictions.csv")
    run("solarfil.annotations", "--data_root", DATA_ROOT, "--out", f"{WORK}/gt_train.csv")
    run("solarfil.evaluate", "--gt", f"{WORK}/gt_train.csv", "--pred", f"{WORK}/oof_predictions.csv", "--bootstrap", 1000, "--plots", "figures")
    run("solarfil.ablate", "--data_root", DATA_ROOT, "--oof_root", WORK, "--cands", f"{WORK}/cands_oof.pkl", "--tau", 0.20, "--boot", 1000)   # Table 1


## 2. Stage 1 on the test photographs, candidates, learned filter

`infer` predicts every test photograph with the ensemble (80 forward passes: 10 networks x 8 flips / rotations), averages the probabilities at 2048 x 2048, extracts the candidate instances and keeps those whose predicted PQ gain reaches the threshold stored with the filter. The probability maps are saved so that the post-processing can be repeated on a CPU (`--from_probs`).

In [ ]:
run("solarfil.infer", "--data_root", DATA_ROOT, "--ckpts", *CKPTS, "--rescorer", RESCORER, "--work", WORK,
    "--tta", TTA, "--save_probs", f"{WORK}/probs", "--out", "submission.csv", *LIMIT)

## 3. One photograph, stage by stage

Left to right, top to bottom: the photograph, the flat-fielded network input, the stage-1 foreground probability, the candidate instances coloured by their learned score, the filter's decision (green kept, red dropped) and the submitted instances.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from solarfil.visualize import pipeline_figure, render_submission

sub = pd.read_csv("submission.csv")
sub["stem"] = sub["filament_id"].str.split("_", n=1).str[0]
stem = sub["stem"].value_counts().index[len(sub["stem"].unique()) // 2]       # a photograph of median richness
img = next(glob.glob(f"{DATA_ROOT}/test/test_images/{stem}.*").__iter__())
display(Image(pipeline_figure(img, f"{WORK}/probs/{stem}.npz", RESCORER, "figures/pipeline_walkthrough.png"), width=1000))

## 4. Submission

Format checks (unique ids, no missing values, masks of one photograph disjoint) run inside `infer`; here a summary and contour overlays of the photographs with most filaments.

In [ ]:
n_img = sub["stem"].nunique()
print(f"submission.csv: {len(sub)} filaments in {n_img} photographs ({len(sub) / max(n_img, 1):.1f} per photograph)")
for f in render_submission("submission.csv", f"{DATA_ROOT}/test/test_images", "figures", n=3)[:3]:
    display(Image(f, width=700))